In [1]:
import base64
import requests
import bs4
import urllib.parse
import json

In [2]:
def get(url):
    api_response = requests.post(
        "https://api.zyte.com/v1/extract",
        auth=("bd4c880861094eff99e91d58573df7ff", ""),
        json={
            "url": url,
            "httpResponseBody": True,
            "followRedirect": True,
        },
    )

    http_response_body: bytes = base64.b64decode(
        api_response.json()["httpResponseBody"]
    )
    return http_response_body.decode("utf-8")

In [3]:
profile_soup = bs4.BeautifulSoup(
    get("https://scholar.google.com/citations?user=qHFA5z4AAAAJ&hl=en&pagesize=100"),
    "html.parser",
)

if profile_soup.select_one("#gs_captcha_f"):
    raise RuntimeError("Google Scholar CAPTCHA")

paper_list = [
    dict(
        {
            "year": row.select_one(".gsc_a_y").get_text(strip=True),
            "citation_url": urllib.parse.urljoin(
                "https://scholar.google.com",
                row.select_one(".gsc_a_at")["href"],
            ),
        }
    )
    for row in profile_soup.select(".gsc_a_tr")
]

print(len(paper_list))
paper_list

57


[{'year': '2023',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Y0pCki6q_DkC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:2osOgNQ5qMEC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:bnK-pcrLprsC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Se3iqnhoufwC'},
 {'year': '2019',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:IjCSPb-OGe4C'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=e

In [4]:
import re
from bs4 import BeautifulSoup


def parse_citation(html):
    soup = BeautifulSoup(
        html.decode("utf-8", "replace") if isinstance(html, bytes) else html,
        "html.parser",
    )

    if soup.select_one("#gs_captcha_f"):
        print("Google Scholar CAPTCHA...")
        return None

    title = soup.select_one("#gsc_oci_title")
    fields = {"title": title.get_text(" ", strip=True) if title else None}
    clusters = []

    for row in soup.select("div.gs_scl"):
        key = row.select_one(".gsc_oci_field")
        value = row.select_one(".gsc_oci_value")
        if not key or not value:
            continue

        name = key.get_text(" ", strip=True).lower()

        if name == "scholar articles":
            for snippet in value.select(".gsc_oci_merged_snippet"):
                link = snippet.select_one("a[href*='cluster=']")
                match = (
                    re.search(r"cluster=(\d+)", link.get("href", "")) if link else None
                )
                if match and match.group(1) not in clusters:
                    clusters.append(match.group(1))
        elif name != "total citations" and name != "description":
            fields[name] = value.get_text(" ", strip=True)

    return {**fields, "clusters": clusters}


for paper in paper_list:
    while True:
        paper_update = parse_citation(get(paper["citation_url"]))
        if not paper_update is None:
            paper |= paper_update
            break
    print(paper["title"], paper["clusters"])

with open("papers2.json", "w") as file:
    json.dump(paper_list, file, indent=4)

Stochastic distributed learning with gradient quantization and double-variance reduction ['10432066948921138844']
Don’t jump through hoops and remove those loops: SVRG and Katyusha are better without the outer loop ['7006814588298832736']
Acceleration for compressed gradient descent in distributed and federated optimization ['1371688885190462102']
From local SGD to local fixed-point methods for federated learning ['15088076385024246021']
RSN: randomized subspace Newton ['7373789986788030402']
Optimal and practical algorithms for smooth and strongly convex decentralized optimization ['15665175903150090179']
Revisiting stochastic extragradient ['8653605232017703121']
Linearly converging error compensated SGD ['9254067822190880000']
A linearly convergent algorithm for decentralized optimization: Sending less bits for free! ['3790088210576421356']
Understanding gradient orthogonalization for deep learning via non-euclidean trust-region optimization ['9464268600160663021']
Accelerated metho